In [0]:
from pyspark.sql.functions import col, to_date, datediff, lit, when, current_timestamp

catalog = "workspace"
schema = "lakehouse_f1"

In [0]:
df_laps = spark.read.table(f"{catalog}.{schema}.bronze_lap_times")
df_races = spark.read.table(f"{catalog}.{schema}.bronze_races")
df_drivers = spark.read.table(f"{catalog}.{schema}.bronze_drivers")
df_pit_stops = spark.read.table(f"{catalog}.{schema}.bronze_pit_stops")
df_results = spark.read.table(f"{catalog}.{schema}.bronze_results")
df_standings = spark.read.table(f"{catalog}.{schema}.bronze_driver_standings")
df_status = spark.read.table(f"{catalog}.{schema}.bronze_status")

In [0]:
df_races_clean = df_races.select(
    col("raceId"),
    col("year").cast("integer").alias("ano_corrida"),
    col("circuitId"),
    to_date(col("date")).alias("data_corrida")
)


In [0]:
df_drivers_clean = df_drivers.select(
    col("driverId"),
    col("driverRef").alias("codigo_piloto"),
    to_date(col("dob")).alias("data_nascimento")
)

In [0]:
df_results_clean = df_results.select(
    col("raceId"),
    col("driverId"),
    col("grid").cast("integer").alias("posicao_largada"),
    col("positionOrder").cast("integer").alias("prosicao_final"),
    col("statusId")
)

In [0]:
df_standings_clean = df_standings.select(
    col("raceId"),
    col("driverId"),
    col("points").cast("integer").alias("pontos_campeonato"),
    col("position").cast("integer").alias("posicao_campeonato")
)

In [0]:
df_status_clean = df_status.select(
    col("statusId"),
    col("status").alias("status_corrida")
)

In [0]:
df_pit_stops_clean = df_pit_stops.select(
    col("raceId"),
    col("driverId"),
    col("stop"),
    col("lap"),
    lit(1).alias("fez_pit_stop")
)

In [0]:
df_silver = df_laps \
    .join(df_races_clean, on="raceId", how="inner") \
    .join(df_drivers_clean, on="driverId", how="inner") \
    .join(df_results_clean, on=["raceId", "driverId"], how="inner") \
    .join(df_standings_clean, on=["raceId", "driverId"], how="inner") \
    .join(df_status_clean, on="statusId", how="inner") \
    .join(df_pit_stops_clean, on=["raceId", "driverId", "lap"], how="left")

In [0]:
df_silver = df_silver \
    .withColumn("fez_pit_stop", when(col("fez_pit_stop").isNull(), 0).otherwise(col("fez_pit_stop"))) \
    .withColumn("idade_piloto_dias", datediff(col("data_corrida"), col("data_nascimento"))) \
    .withColumn("etl_date", current_timestamp())

In [0]:
nome_tabela_silver = f"{catalog}.{schema}.silver_lap_times"
df_silver.write.format("delta").mode("overwrite").saveAsTable(nome_tabela_silver)

print("Camada Silver concluída! Tabela mestre criada com sucesso.")